# 🌳 CART Algorithm — Telecom Churn Prediction
### Decision Tree | Faculty Demo

**Story:** A telecom company wants to know — *which customers are about to leave?*
We load a real CSV file and build a Decision Tree model step by step.

| Section | What We Do |
|---------|-----------|
| **01** | Import Libraries |
| **02** | Load the Dataset |
| **03** | Explore the Data |
| **04** | Prepare Data for Model |
| **05** | Gini Impurity — The Math |
| **06** | Train the CART Model |
| **07** | Check Model Results |
| **08** | See the Decision Tree |
| **09** | Feature Importance |
| **10** | Avoid Overfitting |
| **11** | Final Summary |

> ▶️ Run each cell one by one from top to bottom.


## 📦 Section 01 — Import Libraries

In [ ]:
# If any library is missing, run this line first:
# !pip install scikit-learn pandas matplotlib seaborn

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("All libraries imported successfully!")


## 📂 Section 02 — Load the Dataset

We load the file `telecom_customers.csv` directly using pandas.

> ⚠️ Make sure `telecom_customers.csv` is in the **same folder** as this notebook.

**Columns in the file:**

| Column | What it means |
|--------|--------------|
| customer_id | Unique ID for each customer |
| tenure_months | How many months the customer has been with us |
| monthly_charges | Monthly bill in rupees |
| contract_type | Type of contract (Month-to-Month / 1-Year / 2-Year) |
| tech_support_calls | Number of support calls in last 6 months |
| num_products | How many products they use (1 to 5) |
| online_security | Has online security add-on? (0 = No, 1 = Yes) |
| churn | **Target** — Did the customer leave? (1 = Yes, 0 = No) |


In [ ]:
# Load the CSV file into a DataFrame
df = pd.read_csv("telecom_customers.csv")

# Quick check
print("File loaded successfully!")
print("Total rows    :", len(df))
print("Total columns :", len(df.columns))
print()
print("Column names:")
print(list(df.columns))


In [ ]:
# See the first 8 rows of the data
df.head(8)


In [ ]:
# How many customers churned vs stayed?
churned = df['churn'].sum()
stayed  = len(df) - churned

print("Churned customers :", churned)
print("Stayed  customers :", stayed)
print("Total             :", len(df))
print()
print("Churn rate :", round(churned / len(df) * 100, 1), "%")


## 🔍 Section 03 — Explore the Data

In [ ]:
# Basic info — data types and missing values
print("Data types of each column:")
print(df.dtypes)
print()
print("Missing values in each column:")
print(df.isnull().sum())


In [ ]:
# Summary statistics
df.describe().round(1)


In [ ]:
# Plot 1 — How many customers churned vs stayed?
plt.figure(figsize=(5, 4))
plt.bar(['Stayed', 'Churned'], [stayed, churned], color=['#52B788', '#E63946'])
plt.title('How Many Customers Churned?')
plt.ylabel('Number of Customers')
plt.text(0, stayed  + 1, str(stayed),  ha='center', fontsize=12)
plt.text(1, churned + 1, str(churned), ha='center', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Plot 2 — Churn rate by contract type
contract_churn = df.groupby('contract_type')['churn'].mean() * 100

plt.figure(figsize=(6, 4))
plt.bar(contract_churn.index, contract_churn.values,
        color=['#E63946', '#E9C46A', '#52B788'])
plt.title('Churn Rate by Contract Type (%)')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

print("Churn rate per contract type:")
print(contract_churn.round(1))


In [ ]:
# Plot 3 — Monthly charges for stayed vs churned customers
stayed_charges  = df[df['churn'] == 0]['monthly_charges']
churned_charges = df[df['churn'] == 1]['monthly_charges']

plt.figure(figsize=(7, 4))
plt.hist(stayed_charges,  bins=15, alpha=0.6, color='#52B788', label='Stayed')
plt.hist(churned_charges, bins=15, alpha=0.6, color='#E63946', label='Churned')
plt.title('Monthly Charges: Stayed vs Churned')
plt.xlabel('Monthly Charges (Rs.)')
plt.ylabel('Number of Customers')
plt.legend()
plt.tight_layout()
plt.show()

print("Average charge — Stayed  :", round(stayed_charges.mean(),  1))
print("Average charge — Churned :", round(churned_charges.mean(), 1))


## 🔧 Section 04 — Prepare Data for the Model

Before training, we need to:
1. **Encode** the `contract_type` column — it has text values, the model needs numbers
2. **Separate** features (X) from the target (y)
3. **Split** into training set and test set


In [ ]:
# Step 1 — Encode contract_type (text → number)
# Month-to-Month = 0,  1-Year = 1,  2-Year = 2
contract_map = {'Month-to-Month': 0, '1-Year': 1, '2-Year': 2}

df['contract_type'] = df['contract_type'].map(contract_map)

print("contract_type after encoding:")
print(df['contract_type'].value_counts().sort_index())
print()
print("0 = Month-to-Month,  1 = 1-Year,  2 = 2-Year")


In [ ]:
# Step 2 — Separate features and target
# Drop customer_id (not useful) and churn (that is what we predict)
X = df.drop(columns=['customer_id', 'churn'])
y = df['churn']

print("Feature columns (X):")
print(list(X.columns))
print()
print("Target column (y) = churn")
print("  0 = Stayed,  1 = Churned")


In [ ]:
# Step 3 — Split into 80% training, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size    = 0.2,   # 20% for testing
    random_state = 42     # same split every run
)

print("Training set :", len(X_train), "customers")
print("Test set     :", len(X_test),  "customers")


## 📐 Section 05 — Gini Impurity: The Math Behind CART

CART uses **Gini Impurity** to find the best way to split customers at each node.

### Formula

$$\text{Gini} = 1 - (p_{\text{stayed}}^2 + p_{\text{churned}}^2)$$

| Situation | Gini Value |
|-----------|-----------|
| All customers stayed (pure) | **0.0** |
| All customers churned (pure) | **0.0** |
| 50% stayed, 50% churned (worst) | **0.5** |

### How CART Picks the Best Split

$$\text{Gini Gain} = \text{Gini(before split)} - \left(\frac{N_{\text{left}}}{N} \times \text{Gini(left)} + \frac{N_{\text{right}}}{N} \times \text{Gini(right)}\right)$$

CART tries every possible split and picks the one with the **highest Gini Gain**.


In [ ]:
# Simple Gini function — takes two plain numbers
def calculate_gini(stayed, churned):
    total = stayed + churned
    if total == 0:
        return 0

    p_stayed  = stayed  / total
    p_churned = churned / total

    gini = 1 - (p_stayed**2 + p_churned**2)
    return round(gini, 4)


# Try three easy examples
print("Example 1 — All 10 stayed (pure node):")
print("  Gini =", calculate_gini(stayed=10, churned=0))

print()
print("Example 2 — 5 stayed, 5 churned (totally mixed):")
print("  Gini =", calculate_gini(stayed=5, churned=5))

print()
print("Example 3 — 8 stayed, 2 churned:")
print("  Gini =", calculate_gini(stayed=8, churned=2))


In [ ]:
# Gini Gain example using real telecom numbers
# Before split: 100 customers at the root node
gini_before = calculate_gini(stayed=55, churned=45)

# After splitting on Monthly Charge > Rs.700
gini_left  = calculate_gini(stayed=40, churned=10)   # low charge group
gini_right = calculate_gini(stayed=15, churned=35)   # high charge group

# Weighted average of left and right
weighted = (50 / 100) * gini_left + (50 / 100) * gini_right
gini_gain = round(gini_before - weighted, 4)

print("Before split (100 customers):")
print("  55 stayed, 45 churned  ->  Gini =", gini_before)

print()
print("After split on Monthly Charge > Rs.700:")
print("  Left  (low charge)  : 40 stayed, 10 churned  ->  Gini =", gini_left)
print("  Right (high charge) : 15 stayed, 35 churned  ->  Gini =", gini_right)

print()
print("Gini Gain =", gini_before, "-", round(weighted, 4), "=", gini_gain)
print()
print("CART picks the split with the HIGHEST Gini Gain!")


## 🌳 Section 06 — Train the CART Model

We use `DecisionTreeClassifier` from scikit-learn.

Key settings:
| Parameter | Value | Why |
|-----------|-------|-----|
| `criterion` | `'gini'` | Use Gini Impurity to find splits |
| `max_depth` | `5` | Tree can have at most 5 levels |
| `min_samples_leaf` | `5` | Every leaf needs at least 5 customers |


In [ ]:
# Create the model
model = DecisionTreeClassifier(
    criterion        = 'gini',
    max_depth        = 5,
    min_samples_leaf = 5,
    random_state     = 42
)

# Train on the training data
model.fit(X_train, y_train)

print("Model trained successfully!")
print()
print("Tree depth (number of levels) :", model.get_depth())
print("Number of leaf nodes          :", model.get_n_leaves())


## 📊 Section 07 — Check Model Results

In [ ]:
# Predict on the test set (40 customers the model has never seen)
y_pred = model.predict(X_test)

# Accuracy
acc = accuracy_score(y_test, y_pred)
print("Accuracy:", round(acc * 100, 1), "%")
print()

# Detailed report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Stayed', 'Churned']))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# Read values directly from the matrix
true_negative  = cm[0][0]   # predicted Stayed,  actually Stayed  ✅
false_positive = cm[0][1]   # predicted Churned, actually Stayed  ❌
false_negative = cm[1][0]   # predicted Stayed,  actually Churned ❌
true_positive  = cm[1][1]   # predicted Churned, actually Churned ✅

print("Confusion Matrix Results:")
print("  Correctly predicted STAYED  :", true_negative)
print("  Correctly predicted CHURNED :", true_positive)
print("  Predicted Churned, was Stayed (wrong)  :", false_positive)
print("  Predicted Stayed,  was Churned (wrong) :", false_negative)

# Plot
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stayed', 'Churned'],
            yticklabels=['Stayed', 'Churned'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


In [ ]:
# Training accuracy vs Test accuracy
train_acc = accuracy_score(y_train, model.predict(X_train))
test_acc  = accuracy_score(y_test,  y_pred)

print("Training Accuracy :", round(train_acc * 100, 1), "%")
print("Test Accuracy     :", round(test_acc  * 100, 1), "%")
print("Gap               :", round((train_acc - test_acc) * 100, 1), "%")

if train_acc - test_acc > 0.10:
    print("Note: Large gap — model may be overfitting.")
else:
    print("Note: Small gap — model is generalising well.")

# Bar chart
plt.figure(figsize=(5, 4))
plt.bar(['Training', 'Test'], [train_acc, test_acc],
        color=['#0A9396', '#E9C46A'], width=0.4)
plt.text(0, train_acc + 0.01, str(round(train_acc * 100, 1)) + "%", ha='center', fontsize=12)
plt.text(1, test_acc  + 0.01, str(round(test_acc  * 100, 1)) + "%", ha='center', fontsize=12)
plt.ylim(0, 1.15)
plt.title('Training vs Test Accuracy')
plt.ylabel('Accuracy')
plt.tight_layout()
plt.show()


## 🌲 Section 08 — See the Decision Tree

In [ ]:
# Print tree as text rules — great for explaining in class
feature_names = list(X.columns)

print("Decision Tree Rules (top 3 levels):")
print("=" * 45)
print(export_text(model, feature_names=feature_names, max_depth=3))


In [ ]:
# Draw the tree visually
plt.figure(figsize=(20, 8))
plot_tree(
    model,
    feature_names = feature_names,
    class_names   = ['Stayed', 'Churned'],
    filled        = True,    # colour nodes by majority class
    rounded       = True,
    fontsize      = 9,
    max_depth     = 3        # show top 3 levels so it fits on screen
)
plt.title('CART Decision Tree — Top 3 Levels', fontsize=14)
plt.tight_layout()
plt.show()

print("Blue / green = node has mostly STAYED customers")
print("Orange       = node has mostly CHURNED customers")
print("Darker shade = purer node (lower Gini impurity)")


## 📈 Section 09 — Feature Importance

Which feature was most useful in making predictions?

Each feature gets a score between 0 and 1.
All scores together add up to 1.0.
A higher score means the feature was more important.


In [ ]:
# Get importance scores from the trained model
scores = model.feature_importances_

# Make a simple table
importance_table = pd.DataFrame({
    'Feature'   : feature_names,
    'Importance': scores
})

# Sort highest to lowest
importance_table = importance_table.sort_values('Importance', ascending=False)
importance_table = importance_table.reset_index(drop=True)

print("Feature Importance Scores:")
print()
print(importance_table.to_string(index=False))


In [ ]:
# Bar chart
features_list = importance_table['Feature'].tolist()
scores_list   = importance_table['Importance'].tolist()

plt.figure(figsize=(8, 5))
plt.barh(features_list, scores_list, color='#0A9396')

# Add score labels at the end of each bar
for i in range(len(features_list)):
    plt.text(scores_list[i] + 0.005, i,
             str(round(scores_list[i], 3)),
             va='center', fontsize=10)

plt.xlabel('Importance Score')
plt.title('Which Features Drive Churn the Most?', fontsize=13)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## ✂️ Section 10 — Avoid Overfitting (Depth Tuning)

### What is Overfitting?
A tree that is **too deep** memorises the training data perfectly,
but performs poorly on new customers.

### Solution
Try different values of `max_depth` and pick the one with the best **test accuracy**.


In [ ]:
# Test depths 1 to 10 one by one
train_scores = []
test_scores  = []
depth_values = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

for d in depth_values:
    temp = DecisionTreeClassifier(
        criterion    = 'gini',
        max_depth    = d,
        random_state = 42
    )
    temp.fit(X_train, y_train)

    tr = accuracy_score(y_train, temp.predict(X_train))
    te = accuracy_score(y_test,  temp.predict(X_test))

    train_scores.append(round(tr, 3))
    test_scores.append(round(te, 3))

# Print as a simple table
print("Depth  | Train Acc | Test Acc")
print("-------|-----------|----------")
for i in range(len(depth_values)):
    print(f"  {depth_values[i]:<5}|   {train_scores[i]:.1%}  |  {test_scores[i]:.1%}")


In [ ]:
# Find the best depth
best_acc   = max(test_scores)
best_depth = depth_values[test_scores.index(best_acc)]

print("Best depth     :", best_depth)
print("Best test acc  :", round(best_acc * 100, 1), "%")

# Plot
plt.figure(figsize=(9, 4))
plt.plot(depth_values, train_scores, 'o-', color='#0A9396', label='Train Accuracy')
plt.plot(depth_values, test_scores,  's-', color='#E63946', label='Test Accuracy')
plt.axvline(best_depth, color='gray', linestyle='--',
            label='Best depth = ' + str(best_depth))
plt.xlabel('max_depth')
plt.ylabel('Accuracy')
plt.title('Finding the Right Tree Depth')
plt.legend()
plt.xticks(depth_values)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print()
print("As depth increases, train accuracy keeps going up.")
print("Test accuracy stops improving — that is overfitting.")


In [ ]:
# Retrain the final model using the best depth
final_model = DecisionTreeClassifier(
    criterion        = 'gini',
    max_depth        = best_depth,
    min_samples_leaf = 5,
    random_state     = 42
)
final_model.fit(X_train, y_train)
y_final = final_model.predict(X_test)

final_acc = accuracy_score(y_test, y_final)

print("Final Model")
print("  max_depth :", best_depth)
print("  Accuracy  :", round(final_acc * 100, 1), "%")
print("  Depth     :", final_model.get_depth())
print("  Leaves    :", final_model.get_n_leaves())


## 🏁 Section 11 — Final Summary

In [ ]:
print("=" * 50)
print("  CART DECISION TREE — WHAT WE LEARNED")
print("=" * 50)

print()
print("DATASET")
print("  File    : telecom_customers.csv")
print("  Rows    : 200 customers")
print("  Columns : 6 features + 1 target (churn)")

print()
print("KEY FORMULAS")
print("  Gini = 1 - (p_stayed^2 + p_churned^2)")
print("  Gini Gain = Gini(parent) - weighted Gini(children)")
print("  Higher Gini Gain = Better split")

print()
print("FINAL MODEL PERFORMANCE")
print("  Accuracy  :", round(final_acc * 100, 1), "%")
print("  Depth     :", final_model.get_depth())
print("  Leaves    :", final_model.get_n_leaves())

print()
print("TOP 3 FEATURES THAT DRIVE CHURN")
for i in range(3):
    name  = importance_table.iloc[i]['Feature']
    score = importance_table.iloc[i]['Importance']
    print(f"  {i+1}. {name} (score: {round(score, 3)})")

print()
print("BUSINESS TAKEAWAYS")
print("  1. Month-to-Month customers churn the most")
print("  2. High monthly charges increase churn risk")
print("  3. Many support calls = unhappy customer = likely to leave")


---
## 📌 Quick Revision Card

| Term | Meaning |
|------|---------|
| **Gini Impurity** | Measures how mixed a node is. 0 = pure, 0.5 = fully mixed |
| **Gini Gain** | How much a split reduces impurity. Higher = better split |
| **max_depth** | Maximum number of levels in the tree |
| **Overfitting** | Tree memorises training data but fails on new data |
| **Feature Importance** | Score showing how useful each feature was for prediction |

---
